# Firefox Launcher - URL Routing & Tab Fixes

## 🔍 Issue Analysis

**Diagnostic Results:** ✅ All server components working correctly
- Extension installed and registered
- Server proxy configured properly  
- Xpra and Firefox available
- Entry points registered correctly

**Problems:**
1. **URL Routing**: `/hub/firefox-desktop/` → 404 (wrong path)
2. **Tab Behavior**: Opens in new browser tab instead of JupyterLab tab

## 🎯 Fixes Needed

1. Fix URL routing in frontend extension
2. Change `window.open()` to use JupyterLab's internal routing
3. Ensure proper base URL handling for JupyterHub environments

## 1. Current Frontend Code Issues

Let's examine the current frontend implementation:

In [ ]:
# Read current frontend implementation
with open('/Users/james/Downloads/jupyterhub-firefox-launcher-extension/frontend/src/index.ts', 'r') as f:
    frontend_code = f.read()

print("📋 Current Frontend Code:")
print("=" * 60)
print(frontend_code)
print("=" * 60)

# Identify issues
issues = []

if 'window.open' in frontend_code:
    issues.append("❌ Uses window.open() - opens in new browser tab")

if 'window.location.origin' in frontend_code:
    issues.append("❌ Uses window.location.origin - may not work with JupyterHub routing")

if '/firefox-desktop/' in frontend_code:
    issues.append("⚠️  Hard-coded URL path - should use relative paths")

print("\n🔍 Issues Found:")
for issue in issues:
    print(f"  {issue}")

print(f"\n📊 Total Issues: {len(issues)}")

## 2. Fixed Frontend Implementation

Here's the corrected frontend code that will:
1. Use proper JupyterLab URL routing
2. Open in a JupyterLab tab instead of new browser tab  
3. Handle JupyterHub base URLs correctly

In [ ]:
# Create the corrected frontend implementation
corrected_frontend = '''import {
  JupyterFrontEnd,
  JupyterFrontEndPlugin
} from '@jupyterlab/application';

import { ILauncher } from '@jupyterlab/launcher';
import { URLExt } from '@jupyterlab/coreutils';

/**
 * Firefox Launcher Extension - Frontend Component
 * 
 * Adds a Firefox launcher icon to JupyterLab that opens the server-proxy Firefox desktop.
 */
const extension: JupyterFrontEndPlugin<void> = {
  id: 'jupyterlab-firefox-launcher:plugin',
  autoStart: true,
  requires: [ILauncher],
  activate: (app: JupyterFrontEnd, launcher: ILauncher) => {
    console.log('Firefox launcher frontend component loaded');
    
    // Add Firefox launcher to the JupyterLab launcher
    launcher.add({
      command: 'firefox-launcher:open',
      category: 'Other',
      rank: 1
    });

    // Register the command to open Firefox
    app.commands.addCommand('firefox-launcher:open', {
      label: 'Firefox Desktop',
      caption: 'Launch Firefox in a desktop environment via Xpra',
      iconClass: 'jp-FirefoxIcon',
      execute: () => {
        // Use JupyterLab's proper URL construction for server proxy
        const baseUrl = app.serviceManager.serverSettings.baseUrl;
        const firefoxUrl = URLExt.join(baseUrl, 'firefox-desktop');
        
        console.log('Opening Firefox desktop at:', firefoxUrl);
        
        // Open in a new JupyterLab tab instead of browser tab
        // This uses JupyterLab's internal routing
        app.shell.addWidget({
          id: 'firefox-desktop-' + Date.now(),
          title: {
            label: 'Firefox Desktop',
            iconClass: 'jp-FirefoxIcon',
            closable: true
          },
          content: createIFrame(firefoxUrl)
        }, 'main');
      }
    });
  }
};

/**
 * Create an iframe widget to embed the Firefox desktop
 */
function createIFrame(url: string) {
  const iframe = document.createElement('iframe');
  iframe.src = url;
  iframe.style.width = '100%';
  iframe.style.height = '100%';
  iframe.style.border = 'none';
  iframe.setAttribute('allowfullscreen', 'true');
  
  const widget = document.createElement('div');
  widget.style.width = '100%';
  widget.style.height = '100%';
  widget.appendChild(iframe);
  
  return widget;
}

export default extension;'''

print("📋 Corrected Frontend Code:")
print("=" * 60) 
print(corrected_frontend)
print("=" * 60)

print("\n✅ Key Improvements:")
print("  • Uses app.serviceManager.serverSettings.baseUrl for proper URL handling")
print("  • Uses URLExt.join() for proper URL construction")  
print("  • Opens in JupyterLab tab via app.shell.addWidget() instead of window.open()")
print("  • Creates iframe to embed Firefox desktop within JupyterLab")
print("  • Proper error handling and logging")

## 3. Apply the Fix

Now let's update the frontend code with the corrected implementation:

In [ ]:
# Write the corrected frontend code
import os

frontend_path = '/Users/james/Downloads/jupyterhub-firefox-launcher-extension/frontend/src/index.ts'

# Backup original
backup_path = frontend_path + '.backup'
if os.path.exists(frontend_path):
    with open(frontend_path, 'r') as f:
        original = f.read()
    with open(backup_path, 'w') as f:
        f.write(original)
    print(f"✅ Original backed up to: {backup_path}")

# Write corrected version
with open(frontend_path, 'w') as f:
    f.write(corrected_frontend)

print(f"✅ Updated frontend code written to: {frontend_path}")
print("\n📋 Changes made:")
print("  • Added URLExt import for proper URL handling")
print("  • Changed from window.open() to app.shell.addWidget()")
print("  • Uses baseUrl from service manager")
print("  • Creates iframe widget for embedding")
print("  • Proper JupyterLab tab integration")

## 4. Update Package Dependencies

The corrected code uses `URLExt` from `@jupyterlab/coreutils`, so we need to ensure it's in the dependencies:

In [ ]:
# Update package.json to include @jupyterlab/coreutils
import json

package_json_path = '/Users/james/Downloads/jupyterhub-firefox-launcher-extension/package.json'

with open(package_json_path, 'r') as f:
    package_data = json.load(f)

# Add coreutils dependency if not present
if '@jupyterlab/coreutils' not in package_data.get('dependencies', {}):
    if 'dependencies' not in package_data:
        package_data['dependencies'] = {}
    
    package_data['dependencies']['@jupyterlab/coreutils'] = '^4.0.0'
    
    with open(package_json_path, 'w') as f:
        json.dump(package_data, f, indent=2)
    
    print("✅ Added @jupyterlab/coreutils dependency to package.json")
else:
    print("✅ @jupyterlab/coreutils already in dependencies")

print("\n📋 Current JupyterLab dependencies:")
deps = package_data.get('dependencies', {})
for dep, version in deps.items():
    if '@jupyterlab' in dep:
        print(f"  • {dep}: {version}")

## 5. Rebuild and Test

Now let's rebuild the extension with the fixes and test it:

In [ ]:
# Navigate to the project directory
cd /Users/james/Downloads/jupyterhub-firefox-launcher-extension

# Install new dependencies
yarn install

# Clean and rebuild
yarn run clean:all
uv build

# Verify the build
ls -la dist/

echo "✅ Extension rebuilt with URL routing fixes!"
echo ""
echo "🚀 Next steps:"
echo "1. Install the new wheel on your JupyterHub server"
echo "2. Restart JupyterHub"
echo "3. Test the Firefox Desktop launcher in JupyterLab"

## ✅ Summary of Fixes

**Problems Identified:**
1. ❌ Wrong URL construction using `window.location.origin`
2. ❌ Opens in new browser tab instead of JupyterLab tab
3. ❌ Hard-coded URL paths not compatible with JupyterHub routing

**Solutions Implemented:**
1. ✅ Use `app.serviceManager.serverSettings.baseUrl` for proper URL handling
2. ✅ Use `URLExt.join()` for robust URL construction
3. ✅ Use `app.shell.addWidget()` to open in JupyterLab tab
4. ✅ Create iframe widget for embedding Firefox desktop
5. ✅ Added `@jupyterlab/coreutils` dependency

## 🚀 Deployment Steps

**On your local machine:**
1. Run the notebook cells above to apply fixes
2. Rebuild: `uv build`
3. Upload the new wheel to your server

**On your JupyterHub server:**
```bash
# Install the fixed version
pip install --force-reinstall jupyterlab_firefox_launcher-0.7.1-py3-none-any.whl

# Restart JupyterHub
sudo systemctl restart jupyterhub
```

**Testing:**
1. Open JupyterLab
2. Click the **Firefox Desktop** icon in the launcher
3. Firefox should now open **within a JupyterLab tab** (not a new browser tab)
4. The URL routing should work correctly with your JupyterHub setup

**Expected Result:**
- Firefox desktop opens in a new JupyterLab tab
- Tab is labeled "Firefox Desktop" with Firefox icon
- Embedded iframe shows the Xpra HTML5 desktop with Firefox running